## download_viirs_snpp_peru

In [8]:
import pandas as pd
from datetime import date
from pathlib import Path

In [9]:
def download_viirs_snpp_peru(year: int, save_path: str | None = None) -> tuple[pd.DataFrame, str]:
    """
    Download NASA FIRMS per-country VIIRS-SNPP fires for Peru for a given year.

    Parameters
    ----------
    year : int
        Year to fetch (VIIRS-SNPP is available from ~2012 onward).
    save_path : str | None
        If provided, saves the downloaded CSV to this path.

    Returns
    -------
    (df, url) : (pandas.DataFrame, str)
        The DataFrame with the fires and the exact URL used.

    Example
    -------
    df, url = download_viirs_snpp_peru(2013, "viirs-snpp_2013_Peru.csv")
    """
    # Basic sanity checks
    current_year = date.today().year
    if not isinstance(year, int):
        raise TypeError("year must be an integer, e.g., 2013")
    if year < 2012 or year > current_year:
        raise ValueError(f"year must be between 2012 and {current_year}")

    # URL pattern from your example
    url = f"https://firms.modaps.eosdis.nasa.gov/data/country/viirs-snpp/{year}/viirs-snpp_{year}_Peru.csv"

    # Read directly with pandas
    df = pd.read_csv(url)

    return( df )


In [10]:
fires = download_viirs_snpp_peru(year = 2015)

In [11]:
display(fires)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,-17.20080,-70.95138,297.19,0.72,0.75,2015-01-01,509,N,VIIRS,n,2,286.28,2.81,N,0
1,-17.20262,-70.94351,298.24,0.72,0.75,2015-01-01,509,N,VIIRS,n,2,286.78,2.31,N,0
2,-17.20549,-70.95315,297.41,0.72,0.75,2015-01-01,509,N,VIIRS,n,2,286.69,2.71,N,0
3,-7.10503,-76.51246,310.41,0.47,0.39,2015-01-01,646,N,VIIRS,n,2,289.51,1.21,N,0
4,-17.49601,-71.36060,367.00,0.55,0.68,2015-01-01,1740,N,VIIRS,h,2,295.18,5.94,D,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69991,-3.59187,-80.53916,351.12,0.39,0.36,2015-12-31,1854,N,VIIRS,n,2,303.71,33.86,D,0
69992,-3.57998,-80.48359,337.48,0.39,0.36,2015-12-31,1854,N,VIIRS,n,2,298.99,2.56,D,0
69993,-3.54552,-80.37015,338.71,0.39,0.36,2015-12-31,1854,N,VIIRS,n,2,298.06,2.30,D,0
69994,-3.53509,-80.43648,340.36,0.39,0.36,2015-12-31,1854,N,VIIRS,n,2,299.08,3.51,D,0


## fires_clean

In [12]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from typing import Literal, Optional, Dict, List

In [13]:
def fires_clean(
    df: pd.DataFrame,
    lat_col: str = "latitude",
    lon_col: str = "longitude",
    acq_time_col: str = "acq_time",
    acq_date_col: str = "acq_date",
    crs: str = "EPSG:4326",
) -> gpd.GeoDataFrame:
    """
    Convert raw VIIRS fires DataFrame into a GeoDataFrame and add year/month.

    Assumptions
    ----------
    - The input has columns:
      ['latitude','longitude','bright_ti4','scan','track','acq_date','acq_time',
       'satellite','instrument','confidence','version','bright_ti5','frp','daynight','type']
    - `acq_time` is a DATE string in the form 'YYYY-MM-DD' (per your example).
      If not parseable, we fall back to `acq_date`.

    Returns
    -------
    gdf : GeoDataFrame (EPSG:4326)
        Adds columns: 'event_dt' (Timestamp), 'year' (int), 'month' (int)
    """
    df = df.copy()

    # Coerce coordinates and drop invalid rows
    df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
    df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
    df = df.dropna(subset=[lat_col, lon_col])

    # Parse datetime (prefer acq_time as per your format; fallback to acq_date)
    dt = pd.to_datetime(df[acq_time_col], errors="coerce", utc=False)
    if dt.isna().all() and acq_date_col in df.columns:
        dt = pd.to_datetime(df[acq_date_col], errors="coerce", utc=False)

    if dt.isna().all():
        raise ValueError("Could not parse dates from 'acq_time' or 'acq_date'.")

    df["event_dt"] = dt
    df["year"] = df["event_dt"].dt.year.astype("Int64")
    df["month"] = df["event_dt"].dt.month.astype("Int64")

    # Build geometry
    geom = [Point(xy) for xy in zip(df[lon_col], df[lat_col])]
    gdf = gpd.GeoDataFrame(df, geometry=geom, crs=crs)

    return gdf


In [14]:
gdf_fires = fires_clean(fires)

In [15]:
display(gdf_fires)

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type,event_dt,year,month,geometry
0,-17.20080,-70.95138,297.19,0.72,0.75,2015-01-01,509,N,VIIRS,n,2,286.28,2.81,N,0,1970-01-01 00:00:00.000000509,1970,1,POINT (-70.95138 -17.2008)
1,-17.20262,-70.94351,298.24,0.72,0.75,2015-01-01,509,N,VIIRS,n,2,286.78,2.31,N,0,1970-01-01 00:00:00.000000509,1970,1,POINT (-70.94351 -17.20262)
2,-17.20549,-70.95315,297.41,0.72,0.75,2015-01-01,509,N,VIIRS,n,2,286.69,2.71,N,0,1970-01-01 00:00:00.000000509,1970,1,POINT (-70.95315 -17.20549)
3,-7.10503,-76.51246,310.41,0.47,0.39,2015-01-01,646,N,VIIRS,n,2,289.51,1.21,N,0,1970-01-01 00:00:00.000000646,1970,1,POINT (-76.51246 -7.10503)
4,-17.49601,-71.36060,367.00,0.55,0.68,2015-01-01,1740,N,VIIRS,h,2,295.18,5.94,D,2,1970-01-01 00:00:00.000001740,1970,1,POINT (-71.3606 -17.49601)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69991,-3.59187,-80.53916,351.12,0.39,0.36,2015-12-31,1854,N,VIIRS,n,2,303.71,33.86,D,0,1970-01-01 00:00:00.000001854,1970,1,POINT (-80.53916 -3.59187)
69992,-3.57998,-80.48359,337.48,0.39,0.36,2015-12-31,1854,N,VIIRS,n,2,298.99,2.56,D,0,1970-01-01 00:00:00.000001854,1970,1,POINT (-80.48359 -3.57998)
69993,-3.54552,-80.37015,338.71,0.39,0.36,2015-12-31,1854,N,VIIRS,n,2,298.06,2.30,D,0,1970-01-01 00:00:00.000001854,1970,1,POINT (-80.37015 -3.54552)
69994,-3.53509,-80.43648,340.36,0.39,0.36,2015-12-31,1854,N,VIIRS,n,2,299.08,3.51,D,0,1970-01-01 00:00:00.000001854,1970,1,POINT (-80.43648 -3.53509)


## download_ctry_shp

In [16]:
def download_ctry_shp(iso3 = "PER"):
    ctry = gpd.read_file(r"https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_PER_1.json")
    return ctry

In [17]:
test= download_ctry_shp()

In [18]:
display(test)

,GID_1,GID_0,COUNTRY,NAME_1,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
0,PER.1_1,PER,Peru,Amazonas,NA,NA,Región,Region,01,PE.AM,PE-AMA,"MULTIPOLYGON (((-77.7803 -6.942, -77.7842 -6.9..."
1,PER.2_1,PER,Peru,Ancash,Ancachs,NA,Región,Region,02,PE.AN,PE-ANC,"MULTIPOLYGON (((-77.2862 -10.5582, -77.292 -10..."
2,PER.3_1,PER,Peru,Apurímac,Apuromac,NA,Región,Region,03,PE.AP,NA,"MULTIPOLYGON (((-73.2403 -13.4763, -73.2363 -1..."
3,PER.4_1,PER,Peru,Arequipa,NA,NA,Región,Region,04,PE.AR,PE-ARE,"MULTIPOLYGON (((-71.8424 -17.1726, -71.8482 -1..."
4,PER.5_1,PER,Peru,Ayacucho,NA,NA,Región,Region,05,PE.AY,PE-AYA,"MULTIPOLYGON (((-73.2933 -15.3921, -73.3067 -1..."
5,PER.6_1,PER,Peru,Cajamarca,Caxamarca,NA,Región,Region,06,PE.CJ,PE-CAJ,"MULTIPOLYGON (((-77.8269 -7.3524, -77.8213 -7...."
6,PER.7_1,PER,Peru,Callao,ElCallao,NA,Provincia,Province,07,PE.CL,NA,"MULTIPOLYGON (((-77.2326 -12.1296, -77.2351 -1..."
7,PER.8_1,PER,Peru,Cusco,Cuzco|Qosqo,NA,Región,Region,08,PE.CS,PE-CUS,"MULTIPOLYGON (((-71.0532 -15.4263, -71.0575 -1..."
8,PER.9_1,PER,Peru,Huancavelica,NA,NA,Región,Region,09,PE.HV,PE-HUV,"MULTIPOLYGON (((-74.7775 -14.0364, -74.774 -14..."
9,PER.10_1,PER,Peru,Huánuco,Huknuco,NA,Región,Region,10,PE.HC,NA,"MULTIPOLYGON (((-76.4796 -10.3294, -76.4726 -1..."


In [19]:
ctry = download_ctry_shp(iso3="PER")

In [20]:
display(ctry)

,GID_1,GID_0,COUNTRY,NAME_1,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry
0,PER.1_1,PER,Peru,Amazonas,NA,NA,Región,Region,01,PE.AM,PE-AMA,"MULTIPOLYGON (((-77.7803 -6.942, -77.7842 -6.9..."
1,PER.2_1,PER,Peru,Ancash,Ancachs,NA,Región,Region,02,PE.AN,PE-ANC,"MULTIPOLYGON (((-77.2862 -10.5582, -77.292 -10..."
2,PER.3_1,PER,Peru,Apurímac,Apuromac,NA,Región,Region,03,PE.AP,NA,"MULTIPOLYGON (((-73.2403 -13.4763, -73.2363 -1..."
3,PER.4_1,PER,Peru,Arequipa,NA,NA,Región,Region,04,PE.AR,PE-ARE,"MULTIPOLYGON (((-71.8424 -17.1726, -71.8482 -1..."
4,PER.5_1,PER,Peru,Ayacucho,NA,NA,Región,Region,05,PE.AY,PE-AYA,"MULTIPOLYGON (((-73.2933 -15.3921, -73.3067 -1..."
5,PER.6_1,PER,Peru,Cajamarca,Caxamarca,NA,Región,Region,06,PE.CJ,PE-CAJ,"MULTIPOLYGON (((-77.8269 -7.3524, -77.8213 -7...."
6,PER.7_1,PER,Peru,Callao,ElCallao,NA,Provincia,Province,07,PE.CL,NA,"MULTIPOLYGON (((-77.2326 -12.1296, -77.2351 -1..."
7,PER.8_1,PER,Peru,Cusco,Cuzco|Qosqo,NA,Región,Region,08,PE.CS,PE-CUS,"MULTIPOLYGON (((-71.0532 -15.4263, -71.0575 -1..."
8,PER.9_1,PER,Peru,Huancavelica,NA,NA,Región,Region,09,PE.HV,PE-HUV,"MULTIPOLYGON (((-74.7775 -14.0364, -74.774 -14..."
9,PER.10_1,PER,Peru,Huánuco,Huknuco,NA,Región,Region,10,PE.HC,NA,"MULTIPOLYGON (((-76.4796 -10.3294, -76.4726 -1..."


## intersect_and_collapse_fires_peru

In [13]:
def intersect_and_collapse_fires_peru(
    fires_gdf: gpd.GeoDataFrame,
    peru_lvl2_gdf: gpd.GeoDataFrame,
    id_col: str = "prov_id",
    level: Literal["year", "year_month"] = "year",
    agg_extra: Optional[Dict[str, str]] = None,
    keep_province_attrs: Optional[List[str]] = None,
) -> pd.DataFrame:
    """
    Intersect fire points with Peru Level-2 polygons and collapse counts.

    Parameters
    ----------
    fires_gdf : GeoDataFrame
        Output of `fires_clean` (must contain 'year' and 'month' and a geometry).
    peru_lvl2_gdf : GeoDataFrame
        Peru polygons (e.g., GADM level-2). Must contain an ID column (default 'prov_id').
    id_col : str
        Province identifier column name in `peru_lvl2_gdf`.
    level : 'year' or 'year_month'
        Aggregation level. Default 'year'. If 'year_month', groups by (year, month).
    agg_extra : dict, optional
        Extra aggregations on numeric fields, e.g., {'frp': 'sum'} to sum FRP.
    keep_province_attrs : list[str], optional
        Extra province columns (e.g., ['province','region']) to include in the output.

    Returns
    -------
    df_out : pandas.DataFrame
        Columns include: [id_col], 'n_fires', and time keys ('year' [, 'month']).
        If `keep_province_attrs` provided, they are included (deduplicated merge).
    """
    if fires_gdf.empty:
        # Build empty skeleton with available province ids & time keys if desired
        base = peru_lvl2_gdf[[id_col]].drop_duplicates().copy()
        return base.assign(n_fires=0)

    # Ensure common CRS
    if peru_lvl2_gdf.crs is None:
        raise ValueError("Peru polygons have no CRS. Set to EPSG:4326 or appropriate CRS.")
    if fires_gdf.crs is None:
        raise ValueError("fires_gdf has no CRS. Ensure `fires_clean` was used or set CRS.")

    if peru_lvl2_gdf.crs.to_string() != fires_gdf.crs.to_string():
        fires = fires_gdf.to_crs(peru_lvl2_gdf.crs)
    else:
        fires = fires_gdf

    # Spatial join: point within polygon
    joined = gpd.sjoin(fires, peru_lvl2_gdf[[id_col, peru_lvl2_gdf.geometry.name] + ([] if not keep_province_attrs else keep_province_attrs)],
                       how="inner", predicate="within")

    # Grouping keys
    keys = [id_col]
    if level == "year_month":
        keys += ["year", "month"]
    else:
        keys += ["year"]

    # Count fires + extra aggregations
    agg_dict = {"geometry": "count"}  # we will rename to n_fires
    if agg_extra:
        agg_dict.update(agg_extra)

    grp = joined.groupby(keys, dropna=False).agg(agg_dict).reset_index()
    grp = grp.rename(columns={"geometry": "n_fires"})

    # Optionally attach province attrs (deduplicate first)
    if keep_province_attrs:
        attrs = joined[[id_col] + keep_province_attrs].drop_duplicates(subset=[id_col])
        grp = grp.merge(attrs, on=id_col, how="left")

    # Ensure ints for time keys
    if "year" in grp.columns:
        grp["year"] = grp["year"].astype("Int64")
    if "month" in grp.columns:
        grp["month"] = grp["month"].astype("Int64")

    # Sort nicely
    sort_cols = [c for c in [id_col, "year", "month"] if c in grp.columns]
    grp = grp.sort_values(sort_cols).reset_index(drop=True)

    grpd = gpd.GeoDataFrame(peru_lvl2_gdf.merge(grp, on = id_col))
    return grpd

In [14]:
int_year = intersect_and_collapse_fires_peru(
    gdf_fires, ctry, id_col='GID_1', level="year", agg_extra={"frp": "sum"},
    keep_province_attrs=["NAME_1"]  # optional nice labels
)

In [20]:
display(int_year)

,GID_1,GID_0,COUNTRY,NAME_1_x,VARNAME_1,NL_NAME_1,TYPE_1,ENGTYPE_1,CC_1,HASC_1,ISO_1,geometry,year,n_fires,frp,NAME_1_y
0,PER.1_1,PER,Peru,Amazonas,NA,NA,Región,Region,01,PE.AM,PE-AMA,"MULTIPOLYGON (((-77.7803 -6.942, -77.7842 -6.9...",1970,1256,9890.16,Amazonas
1,PER.2_1,PER,Peru,Ancash,Ancachs,NA,Región,Region,02,PE.AN,PE-ANC,"MULTIPOLYGON (((-77.2862 -10.5582, -77.292 -10...",1970,1335,12456.19,Ancash
2,PER.3_1,PER,Peru,Apurímac,Apuromac,NA,Región,Region,03,PE.AP,NA,"MULTIPOLYGON (((-73.2403 -13.4763, -73.2363 -1...",1970,691,6427.14,Apurímac
3,PER.4_1,PER,Peru,Arequipa,NA,NA,Región,Region,04,PE.AR,PE-ARE,"MULTIPOLYGON (((-71.8424 -17.1726, -71.8482 -1...",1970,534,4267.67,Arequipa
4,PER.5_1,PER,Peru,Ayacucho,NA,NA,Región,Region,05,PE.AY,PE-AYA,"MULTIPOLYGON (((-73.2933 -15.3921, -73.3067 -1...",1970,1366,11583.14,Ayacucho
5,PER.6_1,PER,Peru,Cajamarca,Caxamarca,NA,Región,Region,06,PE.CJ,PE-CAJ,"MULTIPOLYGON (((-77.8269 -7.3524, -77.8213 -7....",1970,2301,20289.56,Cajamarca
6,PER.7_1,PER,Peru,Callao,ElCallao,NA,Provincia,Province,07,PE.CL,NA,"MULTIPOLYGON (((-77.2326 -12.1296, -77.2351 -1...",1970,22,35.53,Callao
7,PER.8_1,PER,Peru,Cusco,Cuzco|Qosqo,NA,Región,Region,08,PE.CS,PE-CUS,"MULTIPOLYGON (((-71.0532 -15.4263, -71.0575 -1...",1970,3513,36665.62,Cusco
8,PER.9_1,PER,Peru,Huancavelica,NA,NA,Región,Region,09,PE.HV,PE-HUV,"MULTIPOLYGON (((-74.7775 -14.0364, -74.774 -14...",1970,618,6138.10,Huancavelica
9,PER.10_1,PER,Peru,Huánuco,Huknuco,NA,Región,Region,10,PE.HC,NA,"MULTIPOLYGON (((-76.4796 -10.3294, -76.4726 -1...",1970,8716,111854.69,Huánuco


## folium_choropleth_peru

In [14]:
import os
import io
import zipfile
from datetime import date
from typing import List, Optional, Tuple

import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium

In [ ]:
def folium_choropleth_peru(
    prov_counts_gdf: gpd.GeoDataFrame,
    value_col: str = "fire_count",
    prov_id: str = "fire_count",
    name_col: str = "province",
    out_html: str = "peru_viirs_fires.html",
) -> folium.Map:
    """
    Builds and saves a Folium choropleth map. Returns the map object.
    """
    # Center roughly on Peru
    m = folium.Map(location=[-9.2, -75.0], zoom_start=5, tiles="cartodbpositron")

    # Prepare choropleth
    chor = folium.Choropleth(
        geo_data=prov_counts_gdf.to_json(),
        data=prov_counts_gdf,
        columns=[prov_id, value_col],
        key_on=f"feature.properties.{prov_id}",
        fill_opacity=0.8,
        line_opacity=0.7,
        legend_name="VIIRS active fires (count)",
        highlight=True,
        nan_fill_opacity=0.2,
        bins=8,
    )
    chor.add_to(m)

    # Add interactive tooltip
    folium.GeoJson(
        prov_counts_gdf.to_json(),
        name="Provinces",
        style_function=lambda x: {"weight": 0.5, "color": "#555", "fillOpacity": 0},
        tooltip=folium.GeoJsonTooltip(
            fields=[name_col, value_col],
            aliases=["Province", "Fires"],
            localize=True,
            sticky=False,
        ),
    ).add_to(m)

    folium.LayerControl().add_to(m)
    m.save(out_html)
    return m

In [17]:
 _map = folium_choropleth_peru(int_year, value_col="n_fires", name_col="NAME_1_x",prov_id='GID_1')